# Building Agentic RAG Systems

This notebook is part of the [Hugging Face Agents Course](https://www.hf.co/learn/agents-course), a free Course from beginner to expert, where you learn to build Agents.

![Agents course share](https://huggingface.co/datasets/agents-course/course-images/resolve/main/en/communication/share.png)

## Install dependencies and configure OpenAI

This notebook follows the same OpenAI setup as `multiagent_notebook.ipynb`. Install `smolagents` with OpenAI support, then set your API key securely (never paste it into the notebook).


In [6]:
# Prefer the agents-course .venv kernel (Kernel -> Change Kernel -> Python 3 / agents-course).
# --no-cache-dir avoids corrupted pip cache warnings.
%pip install -q --no-cache-dir 'smolagents[openai]' langchain langchain-core langchain-text-splitters langchain-community rank_bm25 'pyarrow>=7.0'


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


Set `OPENAI_API_KEY` in your environment or `.env` file. If it is missing, the next cell prompts for it securely with `getpass`.

Optionally set `OPENAI_MODEL` (defaults to `gpt-4o-mini`).


In [2]:
import os
from getpass import getpass

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
print(f"Using OpenAI model: {OPENAI_MODEL}")
print(f"OpenAI is configured: {bool(os.getenv('OPENAI_API_KEY'))}")

Using OpenAI model: gpt-4o-mini
OpenAI is configured: True


## Basic Retrieval with Web Search

Let's build a simple agent that can search the web. This agent will retrieve information and synthesize responses to answer queries. With Agentic RAG, Alfred's agent can:

* Search for latest superhero party trends
* Refine results to include luxury elements
* Synthesize information into a complete plan

This version uses the OpenAI API through `OpenAIServerModel`. Web searches use `WebSearchTool` with the Bing engine (more reliable than the default DuckDuckGo scraper for many environments), plus `VisitWebpageTool` to read promising result pages. No separate SerpAPI or Serper credential is required. The public search backend may rate-limit large runs.

Here's how Alfred's agent can achieve this:


In [3]:
from smolagents import CodeAgent, OpenAIServerModel, VisitWebpageTool, WebSearchTool

# Prefer Bing for more reliable public web search (same pattern as multiagent_notebook.ipynb)
search_tool = WebSearchTool(max_results=10, engine="bing")
visit_tool = VisitWebpageTool()

model = OpenAIServerModel(model_id=OPENAI_MODEL, max_tokens=4096)

agent = CodeAgent(
    model=model,
    tools=[search_tool, visit_tool],
    max_steps=10,
)

# Example usage
response = agent.run(
    "Search for luxury superhero-themed party ideas, including decorations, entertainment, and catering."
)
print(response)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Search for luxury superhero-themed party ideas, including decorations, entertainment, and catering.             │
│                                                                                                                 │
╰─ OpenAIModel - gpt-4o-mini ─────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  party_ideas = web_search(query="luxury superhero-themed party ideas decorations entertainment catering")         
  print(party_ideas)                                                                                               
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
## Search Results

[LUXURY Definition & Meaning - Merriam-Webster](https://www.merriam-webster.com/dictionary/luxury)
The meaning of LUXURY is a condition of abundance or great ease and comfort : sumptuous environment. How to use 
luxury in a sentence.

[LUXURY | English meaning - Cambridge Dictionary](https://dictionary.cambridge.org/dictionary/english/luxury)
LUXURY definition: 1. great comfort, especially as provided by expensive and beautiful things: 2. something 
expensive…. Learn more.

[Luxury - Wikipedia](https://en.wikipedia.org/wiki/Luxury)
This disambiguation page lists articles associated with the title Luxury. If an internal link incorrectly led you 
here, you may wish to change the link to point directly to the intended article.

[Luxury goods - Wikipedia](https://en.wikipedia.org/wiki/Luxury_goods)
In economics, a luxury good (or upmarket good) is a product or item or service for which demand increases more than
what is proportional as income rises, so that expenditures on the good become a more significant proportion of 
overall spending.

[Luxury - Definition, Meaning & Synonyms | Vocabulary.com](https://www.vocabulary.com/dictionary/luxury)
A luxury is something that is pleasant to have or experience, but it is not necessary. For example, eating in a 
restaurant is a luxury — you can always cook and eat at home.

[Luxury Fashion, Designer Wear, Beauty & Lifestyle Shopping Online ...](https://luxury.tatacliq.com/)
Shop curated designer fashion, accessories and lifestyle picks for men & women. Explore top global luxury brands in
watches, bags, footwear and apparel on Tata CLiQ Luxury

[LUXURY | definition in the Cambridge English 
Dictionary](https://dictionary.cambridge.org/us/dictionary/english/luxury)
LUXURY meaning: 1. great comfort, especially as provided by expensive and beautiful things: 2. something 
expensive…. Learn more.

[Luxury (@luxury) • Instagram photos and videos](https://www.instagram.com/luxury/)
5M Followers, 21 Following, 1,544 Posts - Luxury (@luxury) on Instagram: "The world’s leading luxury media platform
Partnerships: 📩 info@igluxury.com"

[Amazon.com: Amazon Luxury Homepage: Amazon Luxury](https://www.amazon.com/b?node=215763371011)
Online shopping for Amazon Luxury Homepage from a great selection at Amazon Luxury Store.

[Luxury | Definition & Examples | Britannica Money](https://www.britannica.com/money/luxury)
luxury, word that implies a relatively large consumption of wealth for nonessential pleasures. There is, however, 
no absolute definition of luxury, for the conception is relative to both time and person.

Out: None

[Step 1: Duration 2.81 seconds| Input tokens: 2,090 | Output tokens: 77]

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 9 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer({                                                                                                   
      "decorations": [                                                                                             
          "Use a color scheme based on popular superhero logos.",                                                  
          "Incorporate superhero-themed tableware and centerpieces.",                                              
          "Set up a backdrop for photo opportunities featuring iconic superhero locations."                        
      ],                                                                                                           
      "entertainment": [                                                                                           
          "Hire actors dressed as popular superheroes for meet-and-greets.",                                       
          "Set up video game stations with superhero-themed games.",                                               
          "Organize superhero-themed contests or scavenger hunts."                                                 
      ],                                                                                                           
      "catering": [                                                                                                
          "Create themed snacks like 'Kryptonite' green jello or 'Hulk' smash cupcakes.",                          
          "Incorporate a signature drink named after a superhero.",                                                
          "Offer a buffet style set up named after different superhero teams."                                     
      ]                                                                                                            
  })                                                                                                               
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: {'decorations': ['Use a color scheme based on popular superhero logos.', 'Incorporate 
superhero-themed tableware and centerpieces.', 'Set up a backdrop for photo opportunities featuring iconic 
superhero locations.'], 'entertainment': ['Hire actors dressed as popular superheroes for meet-and-greets.', 'Set 
up video game stations with superhero-themed games.', 'Organize superhero-themed contests or scavenger hunts.'], 
'catering': ["Create themed snacks like 'Kryptonite' green jello or 'Hulk' smash cupcakes.", 'Incorporate a 
signature drink named after a superhero.', 'Offer a buffet style set up named after different superhero teams.']}

[Step 9: Duration 4.42 seconds| Input tokens: 47,521 | Output tokens: 1,445]

{'decorations': ['Use a color scheme based on popular superhero logos.', 'Incorporate superhero-themed tableware and centerpieces.', 'Set up a backdrop for photo opportunities featuring iconic superhero locations.'], 'entertainment': ['Hire actors dressed as popular superheroes for meet-and-greets.', 'Set up video game stations with superhero-themed games.', 'Organize superhero-themed contests or scavenger hunts.'], 'catering': ["Create themed snacks like 'Kryptonite' green jello or 'Hulk' smash cupcakes.", 'Incorporate a signature drink named after a superhero.', 'Offer a buffet style set up named after different superhero teams.']}


The agent follows this process:

1. **Analyzes the Request:** Alfred’s agent identifies the key elements of the query—luxury superhero-themed party planning, with focus on decor, entertainment, and catering.
2. **Performs Retrieval:** The agent uses `WebSearchTool` (Bing) to find relevant, up-to-date links, then can open pages with `VisitWebpageTool` when more detail is needed.
3. **Synthesizes Information:** After gathering the results, the agent processes them into a cohesive, actionable plan for Alfred, covering all aspects of the party.
4. **Stores for Future Reference:** The agent stores the retrieved information for easy access when planning future events, optimizing efficiency in subsequent tasks.


## Custom Knowledge Base Tool

For specialized tasks, a custom knowledge base can be invaluable. Let's create a tool that queries a vector database of technical documentation or specialized knowledge. Using semantic search, the agent can find the most relevant information for Alfred's needs.

This approach combines predefined knowledge with semantic search to provide context-aware solutions for event planning. With specialized knowledge access, Alfred can perfect every detail of the party.

Dependencies were installed in the first install cell; run the next cell to build the retriever tool.


In [7]:
# langchain-community and rank_bm25 were installed in the setup cell above.
# Re-run that cell if this import fails.
print("Ready to build the custom knowledge-base retriever.")

Ready to build the custom knowledge-base retriever.


In [8]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.retrievers import BM25Retriever
from smolagents import CodeAgent, OpenAIServerModel, Tool

class PartyPlanningRetrieverTool(Tool):
    name = "party_planning_retriever"
    description = "Uses semantic search to retrieve relevant party planning ideas for Alfred’s superhero-themed party at Wayne Manor."
    inputs = {
        "query": {
            "type": "string",
            "description": "The query to perform. This should be a query related to party planning or superhero themes.",
        }
    }
    output_type = "string"

    def __init__(self, docs, **kwargs):
        super().__init__(**kwargs)
        self.retriever = BM25Retriever.from_documents(
            docs, k=5  # Retrieve the top 5 documents
        )

    def forward(self, query: str) -> str:
        assert isinstance(query, str), "Your search query must be a string"

        docs = self.retriever.invoke(
            query,
        )
        return "\nRetrieved ideas:\n" + "".join(
            [
                f"\n\n===== Idea {str(i)} =====\n" + doc.page_content
                for i, doc in enumerate(docs)
            ]
        )

# Simulate a knowledge base about party planning
party_ideas = [
    {"text": "A superhero-themed masquerade ball with luxury decor, including gold accents and velvet curtains.", "source": "Party Ideas 1"},
    {"text": "Hire a professional DJ who can play themed music for superheroes like Batman and Wonder Woman.", "source": "Entertainment Ideas"},
    {"text": "For catering, serve dishes named after superheroes, like 'The Hulk's Green Smoothie' and 'Iron Man's Power Steak.'", "source": "Catering Ideas"},
    {"text": "Decorate with iconic superhero logos and projections of Gotham and other superhero cities around the venue.", "source": "Decoration Ideas"},
    {"text": "Interactive experiences with VR where guests can engage in superhero simulations or compete in themed games.", "source": "Entertainment Ideas"}
]

source_docs = [
    Document(page_content=doc["text"], metadata={"source": doc["source"]})
    for doc in party_ideas
]

# Split the documents into smaller chunks for more efficient search
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    add_start_index=True,
    strip_whitespace=True,
    separators=["\n\n", "\n", ".", " ", ""],
)
docs_processed = text_splitter.split_documents(source_docs)

# Create the retriever tool
party_planning_retriever = PartyPlanningRetrieverTool(docs_processed)

# Reuse the same OpenAI model configuration as the web-search agent
model = OpenAIServerModel(model_id=OPENAI_MODEL, max_tokens=4096)

agent = CodeAgent(tools=[party_planning_retriever], model=model)

# Example usage
response = agent.run(
    "Find ideas for a luxury superhero-themed party, including entertainment, catering, and decoration options."
)

print(response)

/var/folders/tm/7jlz21b97cj8hllswc2xp0kh0000gn/T/ipykernel_15853/147911692.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Find ideas for a luxury superhero-themed party, including entertainment, catering, and decoration options.      │
│                                                                                                                 │
╰─ OpenAIModel - gpt-4o-mini ─────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  party_ideas = party_planning_retriever(query="luxury superhero-themed party ideas including entertainment,       
  catering, and decoration options")                                                                               
  print(party_ideas)                                                                                               
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:

Retrieved ideas:


===== Idea 0 =====
A superhero-themed masquerade ball with luxury decor, including gold accents and velvet curtains.

===== Idea 1 =====
For catering, serve dishes named after superheroes, like 'The Hulk's Green Smoothie' and 'Iron Man's Power Steak.'

===== Idea 2 =====
Decorate with iconic superhero logos and projections of Gotham and other superhero cities around the venue.

===== Idea 3 =====
Hire a professional DJ who can play themed music for superheroes like Batman and Wonder Woman.

===== Idea 4 =====
Interactive experiences with VR where guests can engage in superhero simulations or compete in themed games.

Out: None

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  luxury_superhero_party_ideas = {                                                                                 
      "Entertainment": [                                                                                           
          "Hire a professional DJ who can play themed music for superheroes like Batman and Wonder Woman.",        
          "Interactive experiences with VR where guests can engage in superhero simulations or compete in themed   
  games."                                                                                                          
      ],                                                                                                           
      "Catering": [                                                                                                
          "Serve dishes named after superheroes, like 'The Hulk's Green Smoothie' and 'Iron Man's Power Steak.'"   
      ],                                                                                                           
      "Decoration": [                                                                                              
          "A superhero-themed masquerade ball with luxury decor, including gold accents and velvet curtains.",     
          "Decorate with iconic superhero logos and projections of Gotham and other superhero cities around the    
  venue."                                                                                                          
      ]                                                                                                            
  }                                                                                                                
                                                                                                                   
  final_answer(luxury_superhero_party_ideas)                                                                       
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: {'Entertainment': ['Hire a professional DJ who can play themed music for superheroes like Batman and 
Wonder Woman.', 'Interactive experiences with VR where guests can engage in superhero simulations or compete in 
themed games.'], 'Catering': ["Serve dishes named after superheroes, like 'The Hulk's Green Smoothie' and 'Iron 
Man's Power Steak.'"], 'Decoration': ['A superhero-themed masquerade ball with luxury decor, including gold accents
and velvet curtains.', 'Decorate with iconic superhero logos and projections of Gotham and other superhero cities 
around the venue.']}

[Step 2: Duration 2.22 seconds| Input tokens: 4,405 | Output tokens: 304]

{'Entertainment': ['Hire a professional DJ who can play themed music for superheroes like Batman and Wonder Woman.', 'Interactive experiences with VR where guests can engage in superhero simulations or compete in themed games.'], 'Catering': ["Serve dishes named after superheroes, like 'The Hulk's Green Smoothie' and 'Iron Man's Power Steak.'"], 'Decoration': ['A superhero-themed masquerade ball with luxury decor, including gold accents and velvet curtains.', 'Decorate with iconic superhero logos and projections of Gotham and other superhero cities around the venue.']}


This enhanced agent can:
1. First check the documentation for relevant information
2. Combine insights from the knowledge base
3. Maintain conversation context through memory